Data Isolation & Cleaning

In [37]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import joblib

In [38]:
# 1. Downloading the exact linguistic packages required for text breaking and parsing
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [39]:
# 2. Loading the dataset
df = pd.read_csv('books.csv')
df.head()

,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0
3,9780006178736,0006178731,Rage of angels,NaN,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0
4,9780006280897,0006280897,The Four Loves,NaN,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0


In [40]:
# checking the num of columns and rows (raw)
df.shape

(6810, 12)

In [41]:
# checking for null values (raw)
df.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
subtitle,4429
authors,72
categories,99
thumbnail,329
description,262
published_year,6
average_rating,43


In [42]:
# 3. Isolating only the 9 columns needed for the Frontend UI and Backend NLP
useful_columns = [
    'isbn13', 'isbn10', 'title', 'authors', 'categories',
    'description', 'thumbnail', 'average_rating', 'num_pages'
]
df = df[useful_columns]

In [43]:
# 4. Keeping only the necessary NLP columns and drop rows missing vital data fields
# Only dropping rows where the essential text fields are null.
df = df.dropna(subset=['title', 'authors', 'categories', 'description'])
df.head()

,isbn13,isbn10,title,authors,categories,description,thumbnail,average_rating,num_pages
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,http://books.google.com/books/content?id=KQZCP...,3.85,247.0
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...,http://books.google.com/books/content?id=gA5GP...,3.83,241.0
2,9780006163831,0006163831,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...,http://books.google.com/books/content?id=OmQaw...,3.97,479.0
3,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",http://books.google.com/books/content?id=FKo2T...,3.93,512.0
4,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...,http://books.google.com/books/content?id=XhQ5X...,4.15,170.0


In [44]:
# checking for columns and rows (after dropping)
df.shape

(6446, 9)

In [45]:
# checking for null values (after dropping)
df.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
authors,0
categories,0
description,0
thumbnail,199
average_rating,34
num_pages,34


In [46]:
# Filling in missing UI metadata so the database is uniform and will not crash the UI
df['isbn13'] = df['isbn13'].fillna(0).astype(str)
df['isbn10'] = df['isbn10'].fillna(0).astype(str)
df['thumbnail'] = df['thumbnail'].fillna('https://placehold.co/150x200?text=No+Cover')
df['average_rating'] = df['average_rating'].fillna(0.0)
df['num_pages'] = df['num_pages'].fillna(0)

In [47]:
# checking for null values (final)
df.isnull().sum()

,0
isbn13,0
isbn10,0
title,0
authors,0
categories,0
description,0
thumbnail,0
average_rating,0
num_pages,0


In [48]:
# 5. CATEGORY STANDARDIZATION
def smart_category_split(x):
    cat_str = str(x).strip()

    # If it's a character code, do NOT split at the comma, keep the whole tag!
    if "character" in cat_str.lower() or "fictitious" in cat_str.lower():
        return cat_str

    # Otherwise, perform your original primary genre split
    return cat_str.split(',')[0].strip()

df['categories'] = df['categories'].apply(smart_category_split)

In [49]:
# 6. TARGET VARIABLE PRUNING & ADVANCED CONTENT RESCUE

# 1. Standardize capitalization first to group raw strings cleanly
df['categories'] = df['categories'].str.strip().str.title()

# 2. Advanced Rescue Mapping Layer (Fixes character tags and sub-genres before filtering)
def advanced_genre_rescue(cat_val):
    if pd.isna(cat_val):
        return "Unknown"

    cat_str = str(cat_val).strip()

    # Rule A: Handle character codes (e.g., 'Hyland, Morn (Fictitious character)') -> Fiction
    if "character" in cat_str.lower() or "fictitious" in cat_str.lower():
        return "Fiction"

    # Rule B: Handle sub-genres and variant text layouts mapping to Parent Fiction
    fiction_variants = ['stories', 'tales', 'novel', 'romances', 'humor', 'wit and humor', 'american fiction', 'english fiction']
    if any(variant in cat_str.lower() for variant in fiction_variants):
        if "juvenile" in cat_str.lower():
            return "Juvenile Fiction"
        if "comic" in cat_str.lower() or "graphic" in cat_str.lower():
            return "Comics & Graphic Novels"
        if "poetry" in cat_str.lower():
            return "Poetry"
        if "drama" in cat_str.lower() or "play" in cat_str.lower():
            return "Drama"
        return "Fiction"

    # Rule C: Handle historical, war, and regional tags mapping to Parent History
    history_variants = ['war', 'history', 'century', 'england', 'britain', 'europe', 'united states', 'africa', 'china']
    if any(variant in cat_str.lower() for variant in history_variants):
        if "juvenile" in cat_str.lower():
            return "Juvenile Nonfiction"
        return "History"

    # Rule D: Handle spiritual and theological variations mapping to Parent Religion
    religion_variants = ['religion', 'christian', 'faith', 'church', 'theology', 'bible', 'buddhism']
    if any(variant in cat_str.lower() for variant in religion_variants):
        return "Religion"

    # Rule E: Handle biographical variants mapping to Parent Biography
    if "biography" in cat_str.lower() or "autobiography" in cat_str.lower() or "memoir" in cat_str.lower():
        return "Biography & Autobiography"

    return cat_str

# Apply the rescue mapping rules to the entire dataframe
df['categories'] = df['categories'].apply(advanced_genre_rescue)

# 3. Filter down to the strict 11 parent genres using your frequency threshold
# This runs AFTER rescuing books, so items like Donaldson's novel are safely bundled into 'Fiction'!
genre_counts = df['categories'].value_counts()
dominant_genres = genre_counts[genre_counts >= 75].index
df_optimized = df[df['categories'].isin(dominant_genres)].copy()

print(f" Verification Successful! Total valid books retained for the project: {len(df_optimized)}")
print("\nFinal clean categories:", list(df_optimized['categories'].unique()))

 Verification Successful! Total valid books retained for the project: 4895

Final clean categories: ['Fiction', 'Religion', 'History', 'Juvenile Fiction', 'Biography & Autobiography', 'Philosophy', 'Literary Criticism', 'Juvenile Nonfiction', 'Poetry', 'Comics & Graphic Novels', 'Drama']


In [50]:
# 7. Loading the reference set of English stopwords and initializing the lemmatizer matrix
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_description_text(text):
    # 1. Case Folding: Lowercase everything to avoid duplicated word meanings
    text = str(text).lower()

    # 2. Regex Cleaning: Strip out punctuation, HTML tags, numbers, and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Tokenization: Break the block of text into individual word units
    tokens = word_tokenize(text)

    # 4. Stopword Removal & Lemmatization: Drop useless words (the, is, at)
    # and reduce words to their base form (e.g., "running", "ran" -> "run")
    cleaned_tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    # Re-join clean tokens back into a single whitespace-separated string for vectorization
    return " ".join(cleaned_tokens)

# Run the pipeline over our optimized master dataframe
print("Initializing NLP Preprocessing Pipeline across rows...")
df_optimized['cleaned_description'] = df_optimized['description'].apply(clean_description_text)
print("Pipeline Successfully Completed!")

Initializing NLP Preprocessing Pipeline across rows...
Pipeline Successfully Completed!


In [51]:
# 8. MASTER DATASET EXPORT
# Saving the complete optimized dataframe immediately as the primary UI database
df_optimized.to_pickle('cleaned_books_dataset.pkl')
df_optimized.to_csv('cleaned_books_dataset.csv', index=False)
print("\n Artifact Generated: 'cleaned_books_dataset.pkl' and 'cleaned_books_dataset.csv' is ready for download!")


 Artifact Generated: 'cleaned_books_dataset.pkl' and 'cleaned_books_dataset.csv' is ready for download!


In [53]:
# 9. FINAL VERIFICATION REPORT
print("\n--- VERIFICATION REPORT ---")
print(f"Total Rows: {df_optimized.shape[0]} (Should be 4895)")
print(f"Total Columns: {df_optimized.shape[1]} (Should be 10)")
print("Columns Preset:", list(df_optimized.columns))


--- VERIFICATION REPORT ---
Total Rows: 4895 (Should be 4895)
Total Columns: 10 (Should be 10)
Columns Preset: ['isbn13', 'isbn10', 'title', 'authors', 'categories', 'description', 'thumbnail', 'average_rating', 'num_pages', 'cleaned_description']
